In [0]:
from torn.api import TornAPI
from DBTools.storage import storage
import json
import datetime as dt

header = {"accept": "application/json", "Authorization": f"ApiKey {dbutils.secrets.get('Personal', 'TornAPI')}"}
base_url = "https://api.torn.com/v2/faction/"
base_location = "torn/faction/faction_api_files/"

tables = ["attacks", "balance", "basic", "chains", "crimes", "members", "wars", "news"]

table_params = {
    "attacks": {"sort": "ASC"},
    "balance": {},
    "basic": {},
    "chains": {},
    "crimes": {},
    "members": {},
    "wars": {},
    "news": {"cat": "armoryAction", "sort": "ASC"}
    }

torn_api = TornAPI(key= dbutils.secrets.get('Personal', 'TornAPI'), category="faction")
db_storage = storage(spark)



In [0]:
for table in tables:
    run = True
    params = table_params[table]
    if table =="attacks":
        if spark.catalog.tableExists("torn.faction.faction_attacks"):
            current_data = spark.read.table("torn.faction.faction_attacks")
            max_date = (
                current_data.select("started").agg({"started": "max"}).collect()
            )
            current_max_date = max_date[0]["max(started)"]
        else:
            current_max_date = 1681484237
            
        if (current_max_date<= (dt.datetime.today() + dt.timedelta(days=0)).timestamp()):
            params.update({"from_ts": current_max_date})
        else:
            run = False
    elif table =="chains":
        if spark.catalog.tableExists("torn.faction.faction_chains"):
            current_data = spark.read.table("torn.faction.faction_chains")
            max_date = (
                current_data.select("start").agg({"start": "max"}).collect()
            )
            current_max_date = max_date[0]["max(start)"]
        else:
            current_max_date = 1681484237
    elif table =="balance":
        if spark.catalog.tableExists("torn.faction.faction_balance"):
            current_data = spark.read.table("torn.faction.faction_balance")
            max_date = (
                current_data.select("ts").agg({"ts": "max"}).collect()
            )
            current_max_date = max_date[0]["max(ts)"]
            if current_max_date >= dt.datetime.today().date():
                run = False
    
    table_param = torn_api.prep_params(**params)
    if run:
        fac_json = torn_api.get_json(table,table_param)
        if table == "balance":
            db_storage.store(fac_json, base_location + table, merge_schema=True, add_ts=True)
        else:
            db_storage.store(fac_json, base_location + table, merge_schema=True)

